In [ ]:
from dash import Dash, Input, Output, callback, dash_table
import pandas as pd
import dash_leaflet as dl
import dash_bootstrap_components as dbc
from dash import dcc
from dash import html
import base64
import plotly.express as px
import plotly.graph_objects as go
from animal_shelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################
# update with your username and password and CRUD Python module name
username = "aacuser"
password = "twoDogs@1Cat"
shelter = AnimalShelter(username, password)

# read from the database
df = pd.DataFrame.from_records(shelter.read({}))

# the queries for the custom rescures
water_query    = {"breed":{"$regex":'(Chesa Bay Ret)|(Labrador Retriever)|(Newfoundland)'}, "sex_upon_outcome":"Intact Female", "age_upon_outcome_in_weeks":{"$gte":26,"$lte":156}}
mountain_query = {"breed":{"$regex":'(Shepherd)|(Malamute)|(Sheepdog)|(Husky)|(Rottweiler)'}, "sex_upon_outcome":"Intact Male", "age_upon_outcome_in_weeks":{"$gte":26,"$lte":156}}
disaster_query = {"breed":{"$regex":'(Doberman)|(Shepherd)|(Golden Retriever)|(Bloodhound)|(Rottweiler)'}, "sex_upon_outcome":"Intact Male", "age_upon_outcome_in_weeks":{"$gte":20,"$lte":300}}

# the dataframes for the custom rescues
normalDf = df
waterDf = pd.DataFrame.from_records(shelter.read(water_query))
mountainDf = pd.DataFrame.from_records(shelter.read(mountain_query))
disasterDf = pd.DataFrame.from_records(shelter.read(disaster_query))

# drop the _id and '1' columns from the dataframes
df = df.drop(columns=['_id','1'])
normalDf = normalDf.drop(columns=['_id','1'])
waterDf = waterDf.drop(columns=['_id','1'])
mountainDf = mountainDf.drop(columns=['_id','1'])
disasterDf = disasterDf.drop(columns=['_id','1'])


#########################
# Dashboard Layout / View
#########################

# load logo
image_filename = 'logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())
image = html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                 alt='Grazioso Salvare logo', height=90)

app = Dash(external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    html.Div([
        html.Div(html.B(html.H1('Philip Oriuwa'))),
        html.Div([
            dcc.Link(image, href='https://www.snhu.edu/', target='_blank'),
        ]),
    ],
    id='title',
    style = {'display':'flex', 'align-items':'center', 'justify-content':'center', 'column-gap':'400px'},
    ),
    html.Hr(),
    dcc.RadioItems(['Water Rescue', 'Mountain Rescue', 'Disaster Rescue', 'Reset'], id='rescue-radio'),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": False} for i in df.columns
        ],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_mode='multi',
        row_selectable="single",
        selected_rows=[],
        page_action="native",
        page_current=0,
        page_size=300,
        style_table={'height':'250px', 'overflowX':'auto', 'overflowY':'auto'}
    ),
    html.Hr(),
    html.Div([
        html.Div(id='map-id',style={
            'border-right':'5px solid green', 'padding-right':'1050px',
            'width': '1000px', 'height':'500px'
        }),
        dcc.Graph(id='pie-id'),
    ], style = {
        'display':'flex', 'align-items':'center', 'justify-content':'center', 
        'height':'500px'
    }),
])

#############################################
# Interaction Between Components / Controller
#############################################

# Radio buttons change the underlying dataframe
@app.callback(Output('datatable-id', 'data'), [Input('rescue-radio', 'value')])
def update_table(value):
    if value == 'Water Rescue':
        df = waterDf
    elif value == 'Mountain Rescue':
        df = mountainDf
    elif value == 'Disaster Rescue':
        df = disasterDf
    else:
        df = normalDf
    return df.to_dict('records')
    

# Shows a cursor of the selected animal's location
@app.callback(Output('map-id', 'children'), [Input('datatable-id', 'selected_rows'), Input('rescue-radio', 'value')])
def add_location_map(selected_rows, value):
    # Relevant attributes of the selected animal
    if selected_rows:
        # ensures that the dataframe matches the rescue radio button filter
        if value == 'Water Rescue':
            df = waterDf
        elif value == 'Mountain Rescue':
            df = mountainDf
        elif value == 'Disaster Rescue':
            df = disasterDf
        else:
            df = normalDf
        row_index: int = selected_rows[0]
        lat = df.iloc[selected_rows, 12].at[row_index]
        long = df.iloc[selected_rows, 13].at[row_index]
        animal_name = df.iloc[selected_rows, 8].at[row_index]
        animal_id = df.iloc[selected_rows, 1].at[row_index]
        
        if pd.isna(lat):
            return "Click a populated row "

        # Map with cursor showing the specified animal's location relative to Austin TX [center]
        # top lat = id(A712448)
        # right long = id(A704537)
        # left long = id(A760574)
        # bottom lat = id(A744557)
        center = [30.55,-97.42]

        animal_map = dl.Map(style={'width': '1000px', 'height': '500px'}, center=center, zoom=9, children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(position=[lat, long],
                children=[
                    dl.Tooltip(animal_id),
                    dl.Popup([
                        html.H1(animal_id),
                        html.H2((f'Name: {animal_name}' if animal_name else 'No Name'))
                    ])
                ])
            ])
        return animal_map
    return "Click a row"

# Shows a pie chart of the different animal species selected
@app.callback(Output('pie-id', 'figure'), [Input('datatable-id', 'selected_rows'), Input('rescue-radio', 'value')])
def add_pie_graph(selected_rows, value):
     # ensures that the dataframe matches the rescue radio button filter
    if selected_rows:
        if value == 'Water Rescue':
                df = waterDf
        elif value == 'Mountain Rescue':
            df = mountainDf
        elif value == 'Disaster Rescue':
            df = disasterDf
        else:
            df = normalDf
            
        # array of occurances of an animal's breed 
        breeds = df.iloc[:,3]

        fig = go.Figure(data=[go.Pie(labels=breeds, textinfo='label+percent')])

        fig.update_traces(textinfo='none')
        return fig
    return go.Figure(px.pie(px.data.tips()))

app.run()


Dash is running on http://127.0.0.1:8050/

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8050
Press CTRL+C to quit
127.0.0.1 - - [20/Feb/2023 07:29:57] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "GET /_favicon.ico?v=2.8.0 HTTP/1.1" 200 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "GET /_dash-component-suites/dash/dcc/async-plotlyjs.js HTTP/1.1" 304 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [20/Feb/2023 07:29:58] "POST